# Phase 1: Heart Failure Prediction Project Setup

**Team:** Goblins Hiding in Vents  
**Members:** Cameron, Julian, Juan, Nathan, Nasiru, Jose  
**Goal:** Achieve ≥85% diagnostic accuracy vs hospital baseline of 83.9%  
**Dataset:** UCI Heart Failure Clinical Records (299 patients → expand to 1000 with synthetic data)

## Project Overview

Heart failure affects 64 million people globally with high misdiagnosis rates (16.1%-68.5%). Our machine learning approach using Decision Trees, ROC analysis, and statistical validation aims to improve diagnostic accuracy and save lives.

### Success Metrics
- **Primary Goal:** ≥85% accuracy with 95% confidence
- **Baseline to Beat:** Hospital accuracy of 83.9%
- **Clinical Impact:** Even 1% improvement reduces readmissions and saves lives

### Team Roles
- **Cameron:** Data Collection Lead
- **Julian:** Feature Engineering Specialist  
- **Juan:** Machine Learning Engineer
- **Nathan:** Statistical Analyst
- **Nasiru:** Data Visualization Lead
- **Jose:** Synthetic Data Generator

In [ ]:
# Import required libraries for Phase 1
import pandas as pd
import numpy as np
import requests
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Libraries imported successfully")
print("📊 Starting Phase 1: Data Collection and Project Setup")
print("👥 Team: Goblins Hiding in Vents")

## 1. Project Structure Setup

Let's create the complete directory structure for our 6-phase heart failure prediction project.

In [ ]:
# Create project directory structure
def setup_project_directories():
    """Create complete project structure for 6-phase development."""
    
    directories = [
        "data/raw",
        "data/processed", 
        "data/synthetic",
        "notebooks",
        "models",
        "reports",
        "phase_1_setup",
        "phase_2_eda", 
        "phase_3_features",
        "phase_4_modeling",
        "phase_5_evaluation",
        "phase_6_final"
    ]
    
    base_path = Path("..")
    created_dirs = []
    
    for directory in directories:
        dir_path = base_path / directory
        if not dir_path.exists():
            dir_path.mkdir(parents=True, exist_ok=True)
            created_dirs.append(directory)
        
    return created_dirs

# Setup directories
created = setup_project_directories()
print("📁 Project directory structure:")
for directory in ["data/raw", "data/processed", "data/synthetic", "notebooks", 
                  "models", "reports", "phase_1_setup", "phase_2_eda", 
                  "phase_3_features", "phase_4_modeling", "phase_5_evaluation", 
                  "phase_6_final"]:
    print(f"   └── {directory}")

print(f"\n✅ Created {len(created)} new directories")

## 2. UCI Dataset Collection

Download and validate the UCI Heart Failure Clinical Records dataset (299 patients, 12 features).

In [ ]:
# Download UCI Heart Failure Clinical Records Dataset
def download_uci_dataset():
    """Download UCI heart failure dataset."""
    
    try:
        # Check if we already have the original.csv file
        if Path("../original.csv").exists():
            print("📄 Found existing original.csv file")
            # Copy to proper data directory
            import shutil
            shutil.copy("../original.csv", "../data/raw/heart_failure_clinical_records.csv")
            print("✅ Moved original.csv to data/raw/ directory")
            return True
        
        # If not, download from UCI
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00519/heart_failure_clinical_records_dataset.csv"
        print("🌐 Downloading from UCI repository...")
        
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Save to data directory
        filepath = Path("../data/raw/heart_failure_clinical_records.csv")
        with open(filepath, 'wb') as f:
            f.write(response.content)
        
        print("✅ Dataset downloaded successfully")
        return True
        
    except requests.RequestException as e:
        print(f"❌ Download failed: {e}")
        return False
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return False

# Download the dataset
success = download_uci_dataset()
if success:
    print("📊 Dataset ready for analysis")

In [ ]:
# Load and inspect the dataset
def load_and_inspect_data():
    """Load dataset and perform initial inspection."""
    
    try:
        # Try to load from data directory first
        if Path("../data/raw/heart_failure_clinical_records.csv").exists():
            df = pd.read_csv("../data/raw/heart_failure_clinical_records.csv")
            print("✅ Loaded from data/raw/ directory")
        elif Path("../original.csv").exists():
            df = pd.read_csv("../original.csv")
            print("✅ Loaded from original.csv")
        else:
            print("❌ Dataset file not found")
            return None
        
        print(f"📊 Dataset shape: {df.shape}")
        print(f"📋 Features: {list(df.columns)}")
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None

# Load the dataset
df = load_and_inspect_data()

if df is not None:
    print("\n🔍 First 5 rows:")
    display(df.head())
    
    print("\n📈 Basic dataset info:")
    print(f"   • Total patients: {len(df)}")
    print(f"   • Total features: {len(df.columns)}")
    print(f"   • Missing values: {df.isnull().sum().sum()}")
    print(f"   • Death events: {df['DEATH_EVENT'].sum() if 'DEATH_EVENT' in df.columns else 'N/A'}")

## 3. Data Dictionary and Feature Definitions

Create comprehensive documentation for all 12 clinical features in our heart failure dataset.

In [ ]:
# Create comprehensive data dictionary
def create_data_dictionary():
    """Create detailed data dictionary for all features."""
    
    data_dict = {
        "age": {
            "description": "Age of the patient", 
            "type": "Continuous",
            "unit": "Years",
            "range": "40-95",
            "clinical_relevance": "Older age increases heart failure risk"
        },
        "anaemia": {
            "description": "Decrease of red blood cells or hemoglobin",
            "type": "Binary", 
            "values": "0 = No, 1 = Yes",
            "clinical_relevance": "Can worsen heart failure outcomes"
        },
        "creatinine_phosphokinase": {
            "description": "Level of CPK enzyme in blood",
            "type": "Continuous",
            "unit": "mcg/L", 
            "range": "23-7861",
            "clinical_relevance": "Elevated levels indicate muscle damage"
        },
        "diabetes": {
            "description": "Patient has diabetes mellitus",
            "type": "Binary",
            "values": "0 = No, 1 = Yes", 
            "clinical_relevance": "Major risk factor for heart failure"
        },
        "ejection_fraction": {
            "description": "% of blood leaving heart at each contraction",
            "type": "Continuous",
            "unit": "Percentage",
            "range": "14-80",
            "clinical_relevance": "Key diagnostic measure; <40% indicates HF"
        },
        "high_blood_pressure": {
            "description": "Patient has hypertension",
            "type": "Binary",
            "values": "0 = No, 1 = Yes",
            "clinical_relevance": "Leading cause of heart failure"
        },
        "platelets": {
            "description": "Platelets in the blood",
            "type": "Continuous", 
            "unit": "kiloplatelets/mL",
            "range": "25100-850000",
            "clinical_relevance": "Low levels may indicate bleeding risk"
        },
        "serum_creatinine": {
            "description": "Level of serum creatinine in blood",
            "type": "Continuous",
            "unit": "mg/dL",
            "range": "0.5-9.4", 
            "clinical_relevance": "Kidney function marker; >1.5 indicates dysfunction"
        },
        "serum_sodium": {
            "description": "Level of serum sodium in blood",
            "type": "Continuous",
            "unit": "mEq/L",
            "range": "113-148",
            "clinical_relevance": "Low levels indicate poor HF prognosis"
        },
        "sex": {
            "description": "Patient gender",
            "type": "Binary",
            "values": "0 = Female, 1 = Male",
            "clinical_relevance": "Males have higher HF risk at younger ages"
        },
        "smoking": {
            "description": "Patient smokes",
            "type": "Binary", 
            "values": "0 = No, 1 = Yes",
            "clinical_relevance": "Smoking damages cardiovascular system"
        },
        "time": {
            "description": "Follow-up period",
            "type": "Continuous",
            "unit": "Days",
            "range": "4-285",
            "clinical_relevance": "Longer follow-up provides better outcomes data"
        },
        "DEATH_EVENT": {
            "description": "Patient died during follow-up",
            "type": "Binary",
            "values": "0 = Survived, 1 = Died",
            "clinical_relevance": "TARGET VARIABLE - What we're predicting"
        }
    }
    
    return data_dict

# Create and display data dictionary
data_dictionary = create_data_dictionary()

print("📚 HEART FAILURE DATASET - FEATURE DICTIONARY")
print("=" * 60)

for feature, details in data_dictionary.items():
    print(f"\n🔹 {feature.upper()}")
    print(f"   Description: {details['description']}")
    print(f"   Type: {details['type']}")
    if 'unit' in details:
        print(f"   Unit: {details['unit']}")
    if 'values' in details:
        print(f"   Values: {details['values']}")
    if 'range' in details:
        print(f"   Range: {details['range']}")
    print(f"   Clinical Relevance: {details['clinical_relevance']}")

print("\n" + "=" * 60)
print("✅ Data dictionary created successfully")

## 4. Phase 1 Validation and Next Steps

Validate our Phase 1 setup and prepare for Phase 2 (EDA).

In [ ]:
# Phase 1 completion validation
def validate_phase_1_completion():
    """Validate that Phase 1 objectives are met."""
    
    validation_results = {
        "dataset_loaded": False,
        "correct_shape": False,
        "no_missing_values": False,
        "target_variable_present": False,
        "directories_created": False
    }
    
    # Check if dataset is loaded
    if df is not None:
        validation_results["dataset_loaded"] = True
        
        # Check dataset shape (should be 299 rows, 13 columns)
        if df.shape[0] == 299 and df.shape[1] == 13:
            validation_results["correct_shape"] = True
            
        # Check for missing values
        if df.isnull().sum().sum() == 0:
            validation_results["no_missing_values"] = True
            
        # Check for target variable
        if 'DEATH_EVENT' in df.columns:
            validation_results["target_variable_present"] = True
    
    # Check directories
    required_dirs = ["../data", "../notebooks", "../models", "../reports"]
    if all(Path(d).exists() for d in required_dirs):
        validation_results["directories_created"] = True
    
    return validation_results

# Run validation
validation = validate_phase_1_completion()

print("🔍 PHASE 1 VALIDATION RESULTS")
print("=" * 40)

checks = [
    ("Dataset loaded successfully", validation["dataset_loaded"]),
    ("Correct dataset shape (299x13)", validation["correct_shape"]), 
    ("No missing values", validation["no_missing_values"]),
    ("Target variable present", validation["target_variable_present"]),
    ("Project directories created", validation["directories_created"])
]

for check_name, passed in checks:
    status = "✅" if passed else "❌"
    print(f"{status} {check_name}")

all_passed = all(validation.values())
print("\n" + "=" * 40)
if all_passed:
    print("🎉 PHASE 1 COMPLETE!")
    print("   Ready to proceed to Phase 2: Exploratory Data Analysis")
    print("   Next steps: Statistical summaries, visualizations, correlations")
else:
    print("⚠️  Some validation checks failed. Please review above.")

print("\n📋 PHASE 1 DELIVERABLES SUMMARY:")
print("   • UCI dataset (299 patients, 12 features)")
print("   • Complete project structure")  
print("   • Data dictionary with clinical relevance")
print("   • Team organization and roles")
print("   • Validation and quality checks")